# Product Prototype: Evidence-Based RAG for GRC Tool


**Objective:**

Build a Retrieval-Augmented Generation (RAG) assistant that answers enterprise risk, compliance, and control questions based strictly on approved corporate policies, regulatory frameworks, and control standards.

**Strategic Context:**

Generic Large Language Models (LLMs) introduce unacceptable risk in regulated environments because they may generate incorrect interpretations of policies, controls, or regulatory obligations.

In enterprise environments, Governance, Risk, and Compliance (GRC) requires that AI systems:

*   Act as a controlled policy intelligence layer
*   Reference only approved Sources of Truth (Policies, Standards, Procedures, Framework mappings)
*   Prevent hallucinations or speculative guidance
*   Provide traceable evidence for every response





This prototype demonstrates a Controlled Enterprise GenAI Architecture that retrieves authoritative policy context before generating answers, ensuring alignment with:


*   Internal policies and standards
*   Regulatory requirements (ISO 27001, NIST, SOC 2, GDPR, etc.)
*   Audit and risk governance expectations



**Tech Stack**:

**Orchestration: **LangChain

**Vector Database:** ChromaDB

**Embeddings:** HuggingFace

**Data Source:** Simulated Enterprise Information Security & Compliance Policy (2025)

Cell 1: **Install GenAI Stack**

In [ ]:
# LangChain is the industry standard for building LLM apps (Updated for Latest LangChain v0.3+)
!pip install -q --upgrade requests==2.32.5
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers
!pip install -q opentelemetry-exporter-otlp-proto-http==1.30.0 opentelemetry-api==1.30.0 opentelemetry-sdk==1.30.0

import os

# 1. Text Splitter (New Location)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Embeddings (New Location)
from langchain_huggingface import HuggingFaceEmbeddings

# 3. Vector Store (New Location)
from langchain_chroma import Chroma

# 4. Document Schema
from langchain_core.documents import Document

print("✅ GenAI Environment Configured.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.39.1 requires opentelemetry-exporter-otlp-proto-common==1.39.1, but you have opentelemetry-exporter-otlp-proto-common 1.30.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.39.1 requires opentelemetry-proto==1.39.1, but you have opentelemetry-proto 1.30.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.39.1 requires opentelemetry-sdk~=1.39.1, but you have opentelemetry-sdk 1.30.0 which is incompatible.
opentelemetry-exporter-gcp-lo

In [ ]:
# Splitting the guideline text into smaller chunks for the Vector DB
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_text(guideline_text)

# Wrap text chunks into LangChain Document objects
docs = [Document(page_content=t) for t in texts]

print(f"✅ Split guideline into {len(docs)} document chunks.")

✅ Split guideline into 2 document chunks.


**Cell 2: Ingest "Clinical Guidelines" (Simulated)**

In [ ]:
# Enterprise Scenario: Employees and auditors waste hours searching policies.
# We convert policies and control standards into a queryable knowledge base.

guideline_text = """
POLICY TITLE: Enterprise Information Security & Compliance Policy (2025)

1. ACCESS CONTROL:
All privileged access must follow the principle of least privilege and require formal approval.
User access reviews must be conducted quarterly.

2. INCIDENT MANAGEMENT:
Security incidents must be reported within 1 hour of detection.
All critical incidents require a root cause analysis within 5 business days.

3. DATA PROTECTION:
Sensitive data must be encrypted at rest and in transit using approved encryption standards.
Data classification levels: Public, Internal, Confidential, Restricted.

4. THIRD-PARTY RISK MANAGEMENT:
All vendors handling confidential or restricted data must undergo a security risk assessment annually.

5. AUDIT & LOGGING:
Security logs must be retained for a minimum of 12 months.
Privileged activity must be monitored and reviewed regularly.
"""

**Cell 3: The Brain (Vector Database)**

In [ ]:
# This cell creates the 'db' variable that was missing!

# 1. Initialize the Embedding Model (converts text to numbers)
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create the Vector Database
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_function
)

print("🧠 Knowledge Base Vectorized & Stored in ChromaDB.")

🧠 Knowledge Base Vectorized & Stored in ChromaDB.


**# Cell 4: The Retrieval Logic (No Hallucinations)**

In [ ]:
# PM Insight: We search the DB first, then answer.

def ask_grc_assistant(query):
    # 1. Search the Vector DB for relevant chunks
    results = db.similarity_search(query, k=2)

    # 2. Construct the Evidence
    context = "\n".join([doc.page_content for doc in results])

    print(f"❓ USER QUERY: {query}")
    print("🔎 RETRIEVED POLICY CONTEXT (Source of Truth):")
    print(f"   '{context}'...")
    print("-" * 30)

    # Simulated response logic
    if "least privilege" in context:
        return "All privileged access must follow least privilege and be formally approved. Quarterly access reviews are required."
    elif "1 hour" in context and "incident" in context.lower():
        return "Security incidents must be reported within 1 hour of detection, with root cause analysis for critical incidents."
    elif "encrypted" in context:
        return "Sensitive data must be encrypted at rest and in transit using approved encryption standards."
    else:
        return "No relevant requirement found in approved enterprise policies."

**# Cell 5: Test the Product**

In [ ]:
response = ask_grc_assistant("How quickly must a security incident be reported?")
print(f"🤖 GRC ASSISTANT: {response}")

❓ USER QUERY: How quickly must a security incident be reported?
🔎 RETRIEVED POLICY CONTEXT (Source of Truth):
   'POLICY TITLE: Enterprise Information Security & Compliance Policy (2025)

1. ACCESS CONTROL:
All privileged access must follow the principle of least privilege and require formal approval.
User access reviews must be conducted quarterly.

2. INCIDENT MANAGEMENT:
Security incidents must be reported within 1 hour of detection.
All critical incidents require a root cause analysis within 5 business days.
3. DATA PROTECTION:
Sensitive data must be encrypted at rest and in transit using approved encryption standards.
Data classification levels: Public, Internal, Confidential, Restricted.

4. THIRD-PARTY RISK MANAGEMENT:
All vendors handling confidential or restricted data must undergo a security risk assessment annually.

5. AUDIT & LOGGING:
Security logs must be retained for a minimum of 12 months.
Privileged activity must be monitored and reviewed regularly.'...
---------------

This RAG architecture solves the 'Black Box' problem of LLMs. By forcing the model to retrieve context from a curated Vector Database (Chroma) before answering, we ensure that clinical advice is grounded in approved hospital protocols, not training data hallucinations.